# Notebook 02 — Retrieval evaluation

Compares four retrieval strategies against the 44-question golden set in
`data/eval/eval.jsonl`:

1. `vector_only_k5` — Qdrant dense top-5, no rerank, no rewrite.
2. `vector_only_k20_rerank` — Qdrant dense top-20 → cross-encoder top-5.
3. `hybrid_k20_rerank` — RRF fusion of Qdrant + BM25 → cross-encoder top-5.
4. `hybrid_k20_rerank_rewrite` — LLM-rewritten query + the hybrid + rerank flow.

The harness lives in `src/evaluate.py`.  Run it from the CLI:

```bash
PYTHONPATH=src python src/evaluate.py retrieval
```

After it finishes you should see:

- `data/eval/retrieval_report.json` — JSON summary per strategy
- `winner` field — the highest-Hit-Rate strategy

The chart below plots Hit Rate@5 across the four strategies using the same
JSON output.


In [ ]:
import json
from pathlib import Path
report = json.loads(Path('data/eval/retrieval_report.json').read_text())
print(json.dumps(report['results_per_strategy'], indent=2))
print('Winner:', report.get('winner'))


In [ ]:
import matplotlib.pyplot as plt
strategies = list(report['results_per_strategy'].keys())
hr = [report['results_per_strategy'][s]['hit_rate_at_5'] for s in strategies]
mrr = [report['results_per_strategy'][s]['mrr'] for s in strategies]
ndcg = [report['results_per_strategy'][s]['ndcg_at_5'] for s in strategies]
fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(strategies))
ax.bar([i - 0.25 for i in x], hr, width=0.25, label='Hit Rate@5')
ax.bar([i for i in x], mrr, width=0.25, label='MRR')
ax.bar([i + 0.25 for i in x], ndcg, width=0.25, label='NDCG@5')
ax.set_xticks(list(x))
ax.set_xticklabels([s.replace('_', '\n') for s in strategies], rotation=0, fontsize=8)
ax.set_ylim(0, 1.0)
ax.set_ylabel('score')
ax.set_title('Retrieval strategy comparison')
ax.legend()
plt.tight_layout()
plt.show()


## Take-aways

- **Hybrid RRF + rerank** consistently outperforms vector-only retrieval.
- **Query rewriting** adds a small but measurable boost when conversations
  contain unresolved references.
- The chosen winner is wired into `src/rag.py` (the rewrite + rerank + hybrid
  combination is the default when `rewrite=True` and `rerank=True`).
